# Notebook 04: 双势垒共振隧穿

## 高斯波包入射双势垒的含时演化

---

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import matplotlib.pyplot as plt
from config import SimParams, resonant
from potentials import get_potential
from propagator import propagate
from visualization import (
    set_style, plot_snapshots, plot_TR_evolution,
    plot_momentum_spectrum
)

set_style()

## 1. 双势垒与共振隧穿

双势垒结构：两个方势垒之间形成量子阱

$$V(x) = V_0, \quad x \in [a_1, b_1] \cup [a_2, b_2]$$

**共振隧穿机制**：
1. 量子阱中存在准束缚态（离散能级）
2. 当入射能量与准束缚态能量匹配时，透射率接近 1
3. 能量失配时，透射率远小于单势垒的隧穿率
4. 透射谱呈现尖锐的共振峰

这是共振隧穿二极管 (RTD) 和量子级联激光器的物理基础。

In [ ]:
p = resonant()
print(f'V0 = {p.V0}, k0 = {p.k0}, E_kinetic = {p.E_kinetic:.2f}')
print(f'Barrier 1: [{p.b1_a}, {p.b1_b}], width = {p.b1_b - p.b1_a:.1f}')
print(f'Well: [{p.b1_b}, {p.b2_a}], width = {p.b2_a - p.b1_b:.1f}')
print(f'Barrier 2: [{p.b2_a}, {p.b2_b}], width = {p.b2_b - p.b2_a:.1f}')
print(f'E/V0 = {p.E_kinetic/p.V0:.3f}')

In [ ]:
x = p.x
V = np.real(get_potential(x, p, 'double', with_cap=False))

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(x, V, 'k-', linewidth=2.5)
ax.axhline(p.E_kinetic, color='r', linestyle='--', label=f'E = {p.E_kinetic:.2f}')
ax.set_xlabel('x')
ax.set_ylabel('V(x)')
ax.set_title('Double Barrier Structure')
ax.set_xlim(-10, 10)
ax.legend()
plt.tight_layout()
plt.show()

## 2. 含时演化模拟

In [ ]:
result_double = propagate(p, 'double')

In [ ]:
print(f'Final transmission: T = {result_double.T_values[-1]:.6f}')
print(f'Final reflection: R = {result_double.R_values[-1]:.6f}')
print(f'Norm conservation: int|psi|^2dx = {result_double.norm_values[-1]:.8f}')

### 波包演化快照 — 观察阱内振荡

In [ ]:
fig = plot_snapshots(result_double, n_snapshots=8, x_range=(-50, 50))
plt.show()

### T/R 演化 — 注意阱内概率的延迟释放

In [ ]:
fig = plot_TR_evolution(result_double)
plt.show()

### 动量谱 — 阱内振荡导致动量谱展宽

In [ ]:
fig = plot_momentum_spectrum(result_double, k_range=(-15, 15))
plt.show()

## 3. 透射率 vs 入射能量 — 寻找共振峰

In [ ]:
k0_scan = np.linspace(1.0, 8.0, 30)
T_scan = []

for k0 in k0_scan:
    p_scan = SimParams(V0=10.0, k0=k0, sigma=5.0,
                       b1_a=-3.0, b1_b=-1.0, b2_a=1.0, b2_b=3.0,
                       x0=-25.0)
    res = propagate(p_scan, 'double')
    T_scan.append(res.T_values[-1])
    print(f'k0 = {k0:.2f}, E = {p_scan.E_kinetic:.2f}, T = {res.T_values[-1]:.6f}')

In [ ]:
E_scan = [0.5 * k**2 for k in k0_scan]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(E_scan, T_scan, 'bo-', linewidth=2, markersize=6)
ax.axhline(1.0, color='k', linestyle=':', alpha=0.3)
ax.set_xlabel('E')
ax.set_ylabel('T(E)')
ax.set_title('Double Barrier Transmission vs Incident Energy - Resonance Peaks')
plt.tight_layout()
plt.show()

## 4. 势垒宽度对共振的影响

阱宽度决定准束缚态间距：阱越宽，共振能量越密

In [ ]:
well_widths = [1.0, 2.0, 4.0, 6.0]

fig, ax = plt.subplots(figsize=(10, 6))
for ww in well_widths:
    p_w = SimParams(V0=10.0, k0=4.0, sigma=5.0, x0=-25.0,
                    b1_a=-1.0-ww/2, b1_b=-ww/2,
                    b2_a=ww/2, b2_b=1.0+ww/2)
    k0_s = np.linspace(1.0, 7.0, 25)
    T_s = []
    for k0 in k0_s:
        p_s = SimParams(V0=10.0, k0=k0, sigma=5.0, x0=-25.0,
                        b1_a=-1.0-ww/2, b1_b=-ww/2,
                        b2_a=ww/2, b2_b=1.0+ww/2)
        res = propagate(p_s, 'double')
        T_s.append(res.T_values[-1])
    E_s = [0.5*k**2 for k in k0_s]
    ax.plot(E_s, T_s, '-o', markersize=4, linewidth=1.5, label=f'Well width = {ww:.1f}')

ax.set_xlabel('E')
ax.set_ylabel('T(E)')
ax.set_title('Double Barrier Resonance vs Well Width')
ax.legend()
plt.tight_layout()
plt.show()